# Prepare PyMOL scenes

Create PyMOL scripts with the selected TCR chain as ribbons and the clonotype residues as vdW spheres.

In [ ]:
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "TCRTools").exists() and not (NOTEBOOK_DIR / "data").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "TCRTools"

OUTPUT_DIR = NOTEBOOK_DIR / "output"

In [ ]:
TARGETS_TABLE = OUTPUT_DIR / "resolved_visualization_targets.csv"
targets = pd.read_csv(TARGETS_TABLE)
targets[["tcr_unit_name", "entry_id", "display_chain", "highlight_range"]]

In [ ]:
def write_chain_highlight_pymol_script(structure_path, output_path, chain_id, residue_numbers=None):
    residue_numbers = residue_numbers or []
    lines = [
        "reinitialize",
        f"load {Path(structure_path).as_posix()}, tcr_structure",
        "hide everything",
        "show cartoon, tcr_structure",
        "color gray70, tcr_structure",
        f"color marine, tcr_structure and chain {chain_id}",
        "bg_color white",
        "set cartoon_fancy_helices, on",
        "set ray_opaque_background, off",
    ]
    if residue_numbers:
        residue_selection = "+".join(str(residue) for residue in residue_numbers)
        lines.extend(
            [
                f"select clonotype_residues, tcr_structure and chain {chain_id} and resi {residue_selection}",
                "show spheres, clonotype_residues",
                "color orange, clonotype_residues",
                "set sphere_scale, 0.45, clonotype_residues",
            ]
        )
    lines.extend(["orient", "zoom", "set antialias, 2"])
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text("\n".join(lines) + "\n")
    return output_path

In [ ]:
pymol_scripts = []

for _, target in targets.iterrows():
    residue_numbers = [residue for residue in str(target["highlight_residues"]).split(";") if residue]
    script_path = OUTPUT_DIR / f"{target['sequence_id']}_{target['entry_id']}_{target['display_chain']}.pml"
    pymol_scripts.append(
        write_chain_highlight_pymol_script(
            target["structure_path"],
            script_path,
            target["display_chain"],
            residue_numbers=residue_numbers,
        )
    )

pymol_scripts

In [ ]:
print(pymol_scripts[0].read_text())